## Load the required packages

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["OPENBLAS_NUM_THREADS"] = "16" 
os.environ["MKL_NUM_THREADS"] = "32" 
os.environ["VECLIB_MAXIMUM_THREADS"] = "16" 
os.environ["NUMEXPR_NUM_THREADS"] = "32" 

import sys
sys.path.append(f"{os.getcwd()}/scVAEIT-main/Reproducibility materials")

import numpy as np
import pandas as pd
import scipy.sparse
import tensorflow as tf
import tensorflow_probability as tfp
from types import SimpleNamespace
from scVAEIT.VAEIT import scVAEIT
import anndata
import scipy.io
from pathlib import Path
from scipy.sparse import csr_matrix

tfd = tfp.distributions

2025-10-02 07:02:05.077768: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-02 07:02:05.122723: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-02 07:02:05.122758: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-02 07:02:05.122789: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-02 07:02:05.132082: I tensorflow/core/platform/cpu_feature_g

## Set path

In [2]:
working_dir = Path(os.getcwd())
dataset_dir = f"{working_dir.parents[1]}/datasets/CITE-SLN-Gayoso"
base_dir = f"{working_dir.parents[1]}/outputs/batch variation/208_M2-208_M1/scVAEIT"

## Load data

In [3]:
train_data = anndata.read(f"{dataset_dir}/SLN208_M2.h5ad")
test_data = anndata.read(f"{dataset_dir}/SLN208_M1.h5ad")
train_data, test_data

(AnnData object with n_obs × n_vars = 7105 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 AnnData object with n_obs × n_vars = 8715 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

In [4]:
train_gene_expression = scipy.io.mmread(f"{base_dir}/train_gene_expression.mtx")
test_gene_expression = scipy.io.mmread(f"{base_dir}/test_gene_expression.mtx")
train_gene_expression, test_gene_expression

(<7105x5000 sparse matrix of type '<class 'numpy.int64'>'
 	with 2629429 stored elements in COOrdinate format>,
 <8715x13553 sparse matrix of type '<class 'numpy.int64'>'
 	with 15662727 stored elements in COOrdinate format>)

In [5]:
train_gene_names = pd.read_table(f"{base_dir}/train_HVGs.txt", header=None)
test_gene_names = pd.read_table(f"{base_dir}/test_gene_names.txt", header=None)
train_cell_names = pd.read_table(f"{base_dir}/train_cell_names.txt", header=None)
test_cell_names = pd.read_table(f"{base_dir}/test_cell_names.txt", header=None)

In [6]:
gene_names = np.array(train_gene_names[0].values).astype(str)
ADT_names = np.array(train_data.uns["protein_names"]).astype(str)
gene_names, gene_names.shape, ADT_names, ADT_names.shape

(array(['Lyz2', 'S100a8', 'S100a9', ..., 'Acadm', 'Asf1a', 'Cbx5'],
       dtype='<U13'),
 (5000,),
 array(['ADT_4-1BBLigand(CD137L)_A0890', 'ADT_CD102_A0104',
        'ADT_CD103_A0201', 'ADT_CD105_A0812', 'ADT_CD106_A0226',
        'ADT_CD107a(LAMP-1)_A0905', 'ADT_CD115(CSF-1R)_A0105',
        'ADT_CD117(c-Kit)_A0012', 'ADT_CD11a_A0595', 'ADT_CD11b_A0014',
        'ADT_CD11c_A0106', 'ADT_CD120b(TNFRTypeII-p75)_A0893',
        'ADT_CD122(IL-2Rb)_A0227', 'ADT_CD124(IL-4Ra)_A0916',
        'ADT_CD127(IL-7Ra)_A0198', 'ADT_CD134(OX-40)_A0195',
        'ADT_CD135_A0098', 'ADT_CD137_A0194',
        'ADT_CD138(Syndecan-1)_A0810', 'ADT_CD14_A0424',
        'ADT_CD140a_A0573', 'ADT_CD15(SSEA-1)_A0076',
        'ADT_CD150(SLAM)_A0203', 'ADT_CD152_A0388', 'ADT_CD155(PVR)_A1011',
        'ADT_CD159a(NKG2AB6)_A0927', 'ADT_CD16-32_A0109',
        'ADT_CD160_A1006', 'ADT_CD163_A0417', 'ADT_CD169(Siglec-1)_A0440',
        'ADT_CD170(Singlec-F)_A0431', 'ADT_CD172a(SIRPa)_A0422',
        'ADT_CD178(FasL

## Select common proteins

In [7]:
common_proteins = np.intersect1d(np.array(train_data.uns["protein_names"]), np.array(test_data.uns["protein_names"]))
if isinstance(train_data.obsm["protein_expression"], pd.DataFrame):
    train_protein_expression = train_data.obsm["protein_expression"]
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = train_protein_expression
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)
else:
    train_protein_expression = pd.DataFrame(train_data.obsm["protein_expression"].todense(), index=train_data.obs.index, columns=train_data.uns["protein_names"])
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = csr_matrix(train_protein_expression.values)
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)

if isinstance(test_data.obsm["protein_expression"], pd.DataFrame):
    test_protein_expression = test_data.obsm["protein_expression"]
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = test_protein_expression
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)
else:
    test_protein_expression = pd.DataFrame(test_data.obsm["protein_expression"].todense(), index=test_data.obs.index, columns=test_data.uns["protein_names"])
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = csr_matrix(test_protein_expression.values)
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)

## Set training hyperparameters

In [8]:
dim_input_arr = [len(gene_names), len(ADT_names)]
config = {'dim_input_arr': dim_input_arr, 
          'dimensions':[256],   
          'dim_latent':32,   
          'dim_block': np.array(dim_input_arr), 
          'dist_block':['NB','NB'], 
          'dim_block_enc':np.array([256, 128]),   
          'dim_block_dec':np.array([256, 128]), 
          'dim_block_embed':np.array([32, 16]),   
          'block_names':np.array(['rna', 'adt']), 
          'uni_block_names':np.array(['rna','adt']), 
          'beta_kl':1., 
          'beta_unobs':2./3., 
          'beta_modal':np.array([0.1, 0.9]), 
          'beta_reverse':0.5, 
          "p_feat" : 0.2,   
          "p_modal" : np.ones(2)/2}   
config = SimpleNamespace(**config)

## Normalize reference raw gene expression and protein expression

In [9]:
X = train_gene_expression.toarray()
X = np.log(X / np.sum(X, axis=1, keepdims=True) * 1e4 + 1.)  
X

array([[0.        , 2.0056427 , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 2.69932058, 0.        , ..., 0.        , 0.        ,
        2.69932058],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [10]:
Y = train_protein_expression.values
Y = np.log(Y / np.sum(Y, axis=1, keepdims=True) * 1e4 + 1.)
Y

array([[1.5955048, 3.7896066, 3.4795938, ..., 2.8168013, 3.0279133,
        4.0937157],
       [2.5634737, 4.232531 , 1.6081592, ..., 2.5634737, 2.8317087,
        4.826728 ],
       [3.147665 , 4.4505053, 2.665072 , ..., 1.696776 , 2.9352024,
        4.5494766],
       ...,
       [3.4427376, 4.403379 , 3.266791 , ..., 3.0531344, 2.781067 ,
        4.624072 ],
       [0.       , 4.755833 , 2.696737 , ..., 2.696737 , 3.3555903,
        4.430669 ],
       [0.       , 4.1157284, 3.6403651, ..., 2.3298538, 3.1859846,
        3.8582463]], dtype=float32)

In [11]:
data = np.c_[X, Y]   
data_train_norm = np.exp(data) - 1
data_train_norm, data_train_norm.shape

(array([[  0.        ,   6.43086817,   0.        , ...,  15.72327246,
          19.65408936,  58.96227819],
        [  0.        ,   0.        ,   0.        , ...,  11.98083062,
          15.97443977, 123.80192333],
        [  0.        ,   0.        ,   0.        , ...,   4.45632798,
          17.82531224,  93.5828929 ],
        ...,
        [  0.        ,  13.86962552,   0.        , ...,  20.18163307,
          15.13622742, 100.90816605],
        [  0.        ,   0.        ,   0.        , ...,  13.83125904,
          27.66251993,  82.98757161],
        [  0.        ,   0.        ,   0.        , ...,   9.27643873,
          23.19109551,  46.38218558]]),
 (7105, 5198))

## Initialize network

In [12]:
model = scVAEIT(config, data)

2025-10-02 07:02:49.849767: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6361 MB memory:  -> device: 0, name: Tesla V100-PCIE-16GB, pci bus id: 0000:3b:00.0, compute capability: 7.0


## Start training

In [13]:
model.train(valid=False, num_epoch=300, batch_size=512, save_every_epoch=50, verbose=True, checkpoint_dir=f"{base_dir}/checkpoint")

2025-10-02 07:02:54.173369: E tensorflow/compiler/xla/stream_executor/gpu/asm_compiler.cc:114] *** WARNING *** You are using ptxas 9.1.108, which is older than 11.1. ptxas before 11.1 is known to miscompile XLA code, leading to incorrect results or invalid-address errors.

2025-10-02 07:02:54.173408: E tensorflow/compiler/xla/stream_executor/gpu/asm_compiler.cc:114] *** WARNING *** You are using ptxas 9.1.108, which is older than 11.1. ptxas before 11.1 is known to miscompile XLA code, leading to incorrect results or invalid-address errors.

2025-10-02 07:02:54.173438: E tensorflow/compiler/xla/stream_executor/gpu/asm_compiler.cc:114] *** WARNING *** You are using ptxas 9.1.108, which is older than 11.1. ptxas before 11.1 is known to miscompile XLA code, leading to incorrect results or invalid-address errors.

2025-10-02 07:02:54.173467: E tensorflow/compiler/xla/stream_executor/gpu/asm_compiler.cc:114] *** WARNING *** You are using ptxas 9.1.108, which is older than 11.1. ptxas before

Pretrain - Start of epoch 1


2025-10-02 07:02:57.680868: E tensorflow/compiler/xla/stream_executor/gpu/asm_compiler.cc:114] *** WARNING *** You are using ptxas 9.1.108, which is older than 11.1. ptxas before 11.1 is known to miscompile XLA code, leading to incorrect results or invalid-address errors.

2025-10-02 07:02:57.685745: E tensorflow/compiler/xla/stream_executor/gpu/asm_compiler.cc:114] *** WARNING *** You are using ptxas 9.1.108, which is older than 11.1. ptxas before 11.1 is known to miscompile XLA code, leading to incorrect results or invalid-address errors.

2025-10-02 07:02:57.688417: E tensorflow/compiler/xla/stream_executor/gpu/asm_compiler.cc:114] *** WARNING *** You are using ptxas 9.1.108, which is older than 11.1. ptxas before 11.1 is known to miscompile XLA code, leading to incorrect results or invalid-address errors.

2025-10-02 07:02:57.688775: E tensorflow/compiler/xla/stream_executor/gpu/asm_compiler.cc:114] *** WARNING *** You are using ptxas 9.1.108, which is older than 11.1. ptxas before

14/14 [==============================] - 11s 809ms/step - Reconstructed Loss: 660.5838
Epoch 1, Train ELBO: 803.01, Time elapsed: 0.19 minutes
    rna,     adt,     rna,     adt,      kl
291.58, 130.36, 220.08, 129.42, 31.57
Pretrain - Start of epoch 2
14/14 [==============================] - 3s 236ms/step - Reconstructed Loss: 439.2594
Epoch 2, Train ELBO: 489.75, Time elapsed: 0.24 minutes
    rna,     adt,     rna,     adt,      kl
150.83, 125.60, 78.72, 124.52, 10.07
Pretrain - Start of epoch 3
14/14 [==============================] - 3s 234ms/step - Reconstructed Loss: 377.1613
Epoch 3, Train ELBO: 382.86, Time elapsed: 0.3 minutes
    rna,     adt,     rna,     adt,      kl
68.33, 122.36, 65.82, 122.06,  4.29
Pretrain - Start of epoch 4
14/14 [==============================] - 3s 231ms/step - Reconstructed Loss: 367.0623
Epoch 4, Train ELBO: 369.21, Time elapsed: 0.35 minutes
    rna,     adt,     rna,     adt,      kl
62.06, 121.07, 62.40, 120.98,  2.70
Pretrain - Start of epoch

In [14]:
gene_names_new = np.array(test_gene_names[0].values).astype(str)   
gene_shared = np.intersect1d(gene_names, gene_names_new)   
id_rna = np.array([np.where(gene_names==i)[0][0] for i in gene_shared])   
id_rna_new = np.array([np.where(gene_names_new==i)[0][0] for i in gene_shared])   

ADT_names_new = np.array(test_data.uns["protein_names"]).astype(str)
ADT_shared = np.intersect1d(ADT_names, ADT_names_new)   
id_adt = np.array([np.where(ADT_names==i)[0][0] for i in ADT_shared])   
id_adt_new = np.array([np.where(ADT_names_new==i)[0][0] for i in ADT_shared])

In [15]:
l_rna = np.median(np.sum(data_train_norm[:, id_rna], axis=-1))   

## Fill missing query genes (not in reference) with 0

In [16]:
X_new = test_gene_expression.toarray()
XY_test_norm = np.zeros((X_new.shape[0], len(gene_names)+len(ADT_names)))
XY_test_norm[:, id_rna] = X_new[:, id_rna_new]
XY_test_norm

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

## Normalize query raw gene expression

In [17]:
_XY_test = XY_test_norm.copy()
_XY_test[:, id_rna] = np.log1p(XY_test_norm[:, id_rna] / np.sum(XY_test_norm[:, id_rna], axis=1, keepdims=True) * l_rna)
_XY_test

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [18]:
dataset_test = tf.data.Dataset.from_tensor_slices(
    (
     _XY_test.astype(tf.keras.backend.floatx()), 
     model.cat_enc.transform(np.zeros((_XY_test.shape[0], 1))).toarray().astype(np.float32), 
     np.zeros(_XY_test.shape[0]).astype(np.int32))).batch(512).prefetch(tf.data.experimental.AUTOTUNE)

## Mark the true mask positions in the query

In [19]:
mask = -np.ones((1, XY_test_norm.shape[1]), dtype=tf.keras.backend.floatx())
mask[:, id_rna] = 0.

## Perform prediction on the query

In [20]:
recon = model.vae.get_recon(dataset_test, mask)
Y_hat = recon[:, model.vae.config.dim_input_arr[0]:]   
Y_hat = Y_hat[:, id_adt]
predicted_protein_expression = pd.DataFrame(Y_hat, test_cell_names[0].values, columns=ADT_shared)
predicted_protein_expression.to_csv(f"{base_dir}/test_predicted_protein_expression (scVAEIT).csv")
predicted_protein_expression

,ADT_4-1BBLigand(CD137L)_A0890,ADT_CD102_A0104,ADT_CD103_A0201,ADT_CD105_A0812,ADT_CD106_A0226,ADT_CD107a(LAMP-1)_A0905,ADT_CD115(CSF-1R)_A0105,ADT_CD117(c-Kit)_A0012,ADT_CD11a_A0595,ADT_CD11b_A0014,...,ADT_TCRVr3_A0210,ADT_TCRbchain_A0120,ADT_TCRr-d_A0121,ADT_TER-119-ErythroidCells_A0122,ADT_TIGIT(Vstm3)_A0848,ADT_TLR4(CD284)-MD2Complex_A0875,ADT_Tim-4_A0567,ADT_XCR1_A0568,ADT_anti-P2RY12_A0415,ADT_integrinb7_A0214
AAACCCAAGCGATCGA-1,2.140533,4.289099,4.403958,2.306442,2.429614,1.966511,2.112482,2.904797,5.625441,1.978072,...,3.278938,4.322927,2.480337,0.916291,3.445385,2.296055,2.943569,2.214853,2.697160,5.293599
AAACCCACACCTCTAC-1,1.793322,4.886230,2.725044,1.988842,2.301509,1.990800,1.909684,2.113491,5.024623,2.221970,...,2.948596,2.173235,2.140476,0.769483,3.161398,1.948926,2.740890,1.947510,2.336373,4.050159
AAACCCACACGTCGTG-1,1.835677,5.053160,2.670346,2.017160,2.371835,1.976571,1.973034,2.100354,5.038594,2.381257,...,2.891099,2.252545,2.103234,0.749646,3.108592,1.940380,2.775798,1.913301,2.356596,3.978864
AAACCCACATGTCTAG-1,1.979293,4.996485,2.919373,2.144050,2.399254,2.083879,2.052486,2.276855,4.917598,2.250376,...,3.118991,2.318135,2.354670,0.855173,3.350228,2.102545,2.849922,2.102942,2.563909,4.200674
AAACCCAGTCAAAGTA-1,1.947194,4.931310,2.905937,2.128251,2.401142,2.081375,2.041592,2.271980,4.936630,2.258035,...,3.096137,2.315174,2.322230,0.849802,3.322211,2.080156,2.848489,2.088831,2.532588,4.200497
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,1.927640,4.869174,2.965701,2.116103,2.352002,2.120077,2.013927,2.301383,4.909927,2.165793,...,3.161255,2.419093,2.381220,0.756133,3.381179,2.074101,2.810315,2.111495,2.574726,4.426993
TTTGTTGGTGCTCTTC-1,2.040011,4.868234,3.023052,2.198543,2.437529,2.179743,2.089310,2.357924,4.872181,2.177537,...,3.234538,2.390835,2.447145,0.794128,3.478515,2.127419,2.883712,2.216544,2.675544,4.468607
TTTGTTGTCAGCCTTC-1,1.991730,4.324618,2.922415,2.153900,2.289774,1.885036,2.045850,2.809232,5.780640,2.264163,...,3.182582,4.959318,2.409483,0.642224,3.348657,2.112181,2.848485,2.049859,2.540494,3.948037
TTTGTTGTCCCTTGGT-1,1.934129,4.459838,3.914364,2.182758,2.301955,1.879263,1.970120,2.734278,5.652671,1.906460,...,3.147412,4.175128,2.428051,0.737897,3.298357,2.154602,2.816562,1.997877,2.553436,4.993450
